# 第 2 節｜BERT 二元分類與多類別微調

以 500 筆模擬園遊會報名留言訓練兩個模型。**二元分類**：不能參加（0）、想參加（1）；**三元分類**：不能參加（0）、想參加（1）、尚未確定（2）。二元分類會排除標籤 2 的留言。

請在 Colab 選 **T4 GPU**。首次會下載 BERT 權重，兩次微調可能超過 20 分鐘，實際依 GPU 與網路速度而定。

In [ ]:
# Install the required packages for Hugging Face models.
# 安裝執行 Hugging Face 模型所需的套件。
%pip -q install transformers==4.57.1 datasets==3.6.0 accelerate sentencepiece

In [ ]:
# Import every module used below.
# 集中匯入下方所有程式需要的模組。
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline

## 1. 讀取資料與建立資料分組

CSV 欄位：`message_id` 是編號（不作模型輸入）；`text` 是園遊會留言（X）；`label_3` 是人工設定的分類標籤（y）。標籤 **0＝不能參加、1＝想參加、2＝尚未確定**。資料皆為模擬生成，驗證成績只供課堂理解流程，不代表真實場域效能。

In [ ]:
# Look for the CSV in the Colab /content folder first.
# 先尋找 Colab 的 /content 資料夾中的 CSV。
csv_path = Path("/content/03_校園園遊會留言_500筆.csv")

# Otherwise read the CSV beside this notebook.
# 若不在 /content，就改用目前工作目錄的 CSV。
if not csv_path.exists():
    csv_path = Path("03_校園園遊會留言_500筆.csv")

# Load the examples and their labels.
# 讀取留言和對應的正確標籤。
df = pd.read_csv(csv_path)

# Preview only the fields needed for teaching.
# 只預覽教學需要的欄位。
df.head(3)

In [ ]:
# Split 80 percent of data into the training set.
# 分出 80% 資料作為訓練集。
train_df, rest_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_3"])

# Split the remaining data into validation and test sets.
# 將剩下的資料各分一半，作為驗證集與測試集。
valid_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=42, stratify=rest_df["label_3"])

# Display the three dataset sizes.
# 顯示訓練、驗證、測試資料筆數。
print("Train / Validation / Test:", len(train_df), len(valid_df), len(test_df))

## 2. 選擇模型與教學超參數

修改 `MODEL_ID` 可更換預訓練模型（需支援序列分類）；`BATCH_SIZE` 是一次送入幾筆；`EPOCHS` 是訓練資料重複幾輪；`MAX_STEPS` 控制最多訓練幾個更新 Step（設 `-1` 則以 Epoch 為準）；`LOG_STEPS` 是多久記錄一次 Training Loss；`EVAL_STEPS` 是多久執行 Validation 並儲存 Checkpoint。**Step 指參數更新步數**，不是留言筆數。

**注意：** `MAX_STEPS` 設成正整數會優先於 `EPOCHS`；若設定太小、尚未到 `EVAL_STEPS`，就無法比較足夠的 Checkpoint。

In [ ]:
# Set the pretrained model that will be fine-tuned.
# 設定要微調的預訓練模型 ID。
MODEL_ID = "google-bert/bert-base-chinese"

# Choose how many examples are processed in one batch.
# 設定每次訓練的 batch 筆數。
BATCH_SIZE = 8

# Set the total number of training epochs.
# 設定整份資料要訓練幾輪。
EPOCHS = 2

# Let epochs control training unless a maximum step count is set.
# 預設以 Epoch 決定訓練長度；改成正整數可指定最大 Step。
MAX_STEPS = -1

# Record training loss every few optimizer steps.
# 設定每隔幾個更新 Step 記錄 Training Loss。
LOG_STEPS = 5

# Evaluate and save a checkpoint every few steps.
# 設定每隔幾個 Step 進行驗證並儲存 Checkpoint。
EVAL_STEPS = 10

In [ ]:
# Load the tokenizer matching the model ID.
# 載入與指定模型相同的 Tokenizer。
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Define one simple tokenization step.
# 定義一次文字轉換成 Token ID 的操作。
def encode(texts):
    # Give every text the same short length for this activity.
    # 這次練習把每句留言統一補齊或截短為 48 個 Token。
    return tokenizer(texts.tolist(), truncation=True, padding="max_length", max_length=48)

# Turn text into input IDs and attention masks.
# 將留言轉成模型需要的 Token ID 與 Attention Mask。
tokens = encode(df["text"])

# Attach the token IDs to each original example.
# 將轉換後的 Token ID 加入每筆原始資料。
df["input_ids"] = tokens["input_ids"]

# Keep which token positions contain real text.
# 保留哪些位置是真正文字的 Attention Mask。
df["attention_mask"] = tokens["attention_mask"]

**提醒：**剛才先切分資料的列，接下來需要在相同原始列上取得 Token。為避免索引對錯，下面會根據 `message_id` 將處理後的欄位併回三組資料。

In [ ]:
# Match tokenized examples to the original message IDs.
# 使用留言編號將 Token 資料對應回原始資料。
train_df = df.loc[train_df.index].copy()

# Restore the tokenized validation examples.
# 取回已轉換的驗證集留言。
valid_df = df.loc[valid_df.index].copy()

# Restore the tokenized test examples.
# 取回已轉換的測試集留言。
test_df = df.loc[test_df.index].copy()

# Give Trainer the label column name it expects.
# 將正確答案欄位命名為 Trainer 使用的 labels。
train_df["labels"] = train_df["label_3"]
valid_df["labels"] = valid_df["label_3"]
test_df["labels"] = test_df["label_3"]

## 3. 二元分類（2 個標籤）

只保留「不能參加（0）／想參加（1）」，排除「尚未確定（2）」。`Trainer` 會根據訓練資料更新 BERT Encoder 和新的分類頭。

In [ ]:
# Keep only the two definite attendance labels.
# 只保留明確不能參加或想參加的兩個標籤。
train_bin = Dataset.from_pandas(train_df[train_df["labels"] != 2], preserve_index=False)

# Build the binary validation set the same way.
# 使用相同規則建立二元分類驗證集。
valid_bin = Dataset.from_pandas(valid_df[valid_df["labels"] != 2], preserve_index=False)

# Reserve the binary test set for the final evaluation.
# 保留二元分類測試集供最後評估。
test_bin = Dataset.from_pandas(test_df[test_df["labels"] != 2], preserve_index=False)

# Add a two-class classification head to pretrained BERT.
# 在預訓練 BERT 上加入輸出兩個類別的分類頭。
binary_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2, id2label={0:"不能參加", 1:"想參加"})

### 設定 Trainer

這裡只保留與教學相關的參數，以及**每隔固定 Step 驗證並儲存 Checkpoint** 所必需的設定。`load_best_model_at_end=True` 會在訓練結束時載入 Validation Loss 最低的模型。

In [ ]:
# Set training steps, evaluation, and checkpoint selection.
# 設定訓練輪數、驗證頻率，以及最佳 Checkpoint 的選擇方式。
args_binary = TrainingArguments(
    output_dir="03_binary_checkpoints", num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, max_steps=MAX_STEPS, learning_rate=3e-5,
    logging_steps=LOG_STEPS, eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=EVAL_STEPS,
    load_best_model_at_end=True, save_total_limit=2, report_to="none"
)

# Connect the model, settings, and labeled datasets.
# 將模型、訓練設定、訓練集及驗證集交給 Trainer。
binary_trainer = Trainer(model=binary_model, args=args_binary,
                         train_dataset=train_bin, eval_dataset=valid_bin)

In [ ]:
# Fine-tune the binary classifier.
# 開始微調二元分類模型。
binary_trainer.train()

# Show the checkpoint chosen by the lowest validation loss.
# 顯示根據最低 Validation Loss 選出的最佳 Checkpoint。
print("Best checkpoint:", binary_trainer.state.best_model_checkpoint)
print("Best Validation Loss:", round(binary_trainer.state.best_metric, 4))

In [ ]:
# Convert real Trainer logs into a table.
# 將實際訓練紀錄轉成表格。
history = pd.DataFrame(binary_trainer.state.log_history)

# Draw the training loss against optimizer steps.
# 以更新 Step 為橫軸繪製 Training Loss。
ax = history.dropna(subset=["loss"]).plot(x="step", y="loss", label="Training Loss")

# Overlay the validation loss from the checkpoints.
# 在同一張圖上加上各次驗證的 Validation Loss。
history.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o", label="Validation Loss")

# Display both curves together.
# 顯示兩條 Loss 曲線。
plt.show()

In [ ]:
# Evaluate the selected binary model on untouched test data.
# 使用保留的測試集評估已選定的二元模型。
binary_test = binary_trainer.predict(test_bin)

# Show just the final test accuracy.
# 僅顯示最後測試集的 Accuracy。
print("Binary Test Accuracy:", round(accuracy_score(binary_test.label_ids, binary_test.predictions.argmax(axis=1)), 3))

In [ ]:
# Create an inference pipeline from the selected binary model.
# 使用自動載入的最佳二元模型建立預測工具。
binary_pipe = pipeline("text-classification", model=binary_trainer.model, tokenizer=tokenizer)

# Try three new messages that were not in the CSV.
# 測試三句沒有出現在 CSV 的新留言。
binary_texts = ["我已確定週末能到學校幫忙。", "很抱歉，那天我得去上課，不能參加。", "園遊會的活動我會準時到場！"]

# Predict and display each attendance decision.
# 依序預測並顯示三句留言的分類結果。
for text, result in zip(binary_texts, binary_pipe(binary_texts)):
    print(text, "→", result["label"], round(result["score"], 3))

## 4. 多類別分類（3 個標籤）

這次改為三個類別：**不能參加（0）／想參加（1）／尚未確定（2）**。重新從同一個預訓練 BERT 建立**三類分類頭**，不是把剛才二元分類頭改名稱。

In [ ]:
# Release the binary classifier to save GPU memory.
# 先釋放二元分類模型，降低第二次訓練的 GPU 記憶體需求。
del binary_pipe, binary_trainer, binary_model

# Clear unused GPU allocations when CUDA is available.
# 若有 CUDA，清理目前未使用的 GPU 記憶體。
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Keep all three classes for the next task.
# 多類別分類保留全部三種標籤。
train_multi = Dataset.from_pandas(train_df, preserve_index=False)
valid_multi = Dataset.from_pandas(valid_df, preserve_index=False)
test_multi = Dataset.from_pandas(test_df, preserve_index=False)

# Create a new BERT classifier with three output classes.
# 重新載入 BERT，建立具有三個輸出類別的分類頭。
multi_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=3, id2label={0:"不能參加", 1:"想參加", 2:"尚未確定"})

In [ ]:
# Reuse the same learning settings in a separate output folder.
# 在另一個輸出目錄使用相同超參數訓練三元分類。
args_multi = TrainingArguments(
    output_dir="03_multiclass_checkpoints", num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, max_steps=MAX_STEPS, learning_rate=3e-5,
    logging_steps=LOG_STEPS, eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=EVAL_STEPS,
    load_best_model_at_end=True, save_total_limit=2, report_to="none"
)

# Create a new Trainer with the three-class datasets.
# 使用三元分類訓練集與驗證集建立新的 Trainer。
multi_trainer = Trainer(model=multi_model, args=args_multi,
                        train_dataset=train_multi, eval_dataset=valid_multi)

In [ ]:
# Fine-tune BERT for three-class classification.
# 開始微調 BERT，使它能分類三種報名狀態。
multi_trainer.train()

# Show the best model selected by validation loss.
# 顯示 Validation Loss 最低的 Checkpoint。
print("Best checkpoint:", multi_trainer.state.best_model_checkpoint)
print("Best Validation Loss:", round(multi_trainer.state.best_metric, 4))

In [ ]:
# Collect the real training logs for the three-class model.
# 收集三元分類模型實際產生的訓練紀錄。
history = pd.DataFrame(multi_trainer.state.log_history)

# Plot the training loss from each logged step.
# 繪製每次記錄的 Training Loss。
ax = history.dropna(subset=["loss"]).plot(x="step", y="loss", label="Training Loss")

# Plot validation loss on the same chart.
# 在同一張圖上加入 Validation Loss。
history.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o", label="Validation Loss")

# Show the loss curves.
# 顯示兩條真實的 Loss 曲線。
plt.show()

In [ ]:
# Evaluate the selected three-class model on its test set.
# 使用保留的測試集評估已選定的三元分類模型。
multi_test = multi_trainer.predict(test_multi)

# Display final test accuracy.
# 顯示三元分類測試集的 Accuracy。
print("Multiclass Test Accuracy:", round(accuracy_score(multi_test.label_ids, multi_test.predictions.argmax(axis=1)), 3))

In [ ]:
# Create a predictor from the best three-class model.
# 使用最佳三元分類模型建立預測工具。
multi_pipe = pipeline("text-classification", model=multi_trainer.model, tokenizer=tokenizer)

# Prepare three new examples covering all three intended outcomes.
# 準備三句新留言，分別代表不同的報名意向。
multi_texts = ["我會提早到場準備攤位，請算我一份。", "那天我要去看醫生，真的無法參加。", "我可能可以去，不過還要先問家人。"]

# Show each predicted class and model score.
# 顯示每句話的預測類別及分數。
for text, result in zip(multi_texts, multi_pipe(multi_texts)):
    print(text, "→", result["label"], round(result["score"], 3))

**課堂觀察：**比較二元與三元分類的 `num_labels`、Test Accuracy，以及兩張真實 Loss 圖。`best_model_checkpoint` 是這次執行真正選出的路徑，不是事先指定的答案。**驗證 Loss 下降也不保證真實應用表現**；本資料為人工合成且句型有相似性，模型結果可能偏樂觀。